# Device Code login → a **Lance** table

Sign in with the **OAuth2 Device Authorization Grant** (RFC 8628) — you approve a code in a
browser on any device, ideal for CLIs and remote kernels — then create a Lance generic table
and round-trip an embeddings dataset through it with vended S3 credentials.

> **Kernel:** run this with the **Python (pylakekeeper)** kernel (the `python/.venv`). If you
> see `ModuleNotFoundError: No module named 'pylakekeeper'`, you're on the wrong kernel — see
> `examples/README.md` → *Running the notebooks*.

## Requirements
```sh
cd python && pip install -e '.[examples]'   # pylakekeeper + lance + pyarrow + numpy
```
A Keycloak-fronted Lakekeeper (e.g. the `access-control-advanced` stack) and an existing
warehouse + namespace (see `examples/README.md`).

In [ ]:
import os

KEYCLOAK = os.environ.get("KEYCLOAK", "http://localhost:30080")
REALM = os.environ.get("KEYCLOAK_REALM", "iceberg")
ISSUER = f"{KEYCLOAK}/realms/{REALM}"
DEVICE_AUTHORIZATION_URL = f"{ISSUER}/protocol/openid-connect/auth/device"
TOKEN_URL = f"{ISSUER}/protocol/openid-connect/token"
CLIENT_ID = os.environ.get("OAUTH_CLIENT_ID", "lakekeeper")  # public client, no secret
SCOPE = os.environ.get("OAUTH_SCOPE", "openid offline_access")

# --- Lakekeeper ---
LAKEKEEPER = os.environ.get("LAKEKEEPER", "http://localhost:8181")
WAREHOUSE_ID = os.environ.get("WAREHOUSE_ID", "")  # the warehouse UUID (URL prefix, not name)
PROJECT_ID = os.environ.get("PROJECT_ID") or None
NAMESPACE = os.environ.get("NAMESPACE", "ai.test")

In [ ]:
import base64
import json
import time


def token_seconds_left(auth_header: str) -> int:
    """Decode a Bearer JWT's `exp` claim and return seconds until it expires."""
    token = auth_header.split(" ", 1)[-1]
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)  # restore base64 padding
    claims = json.loads(base64.urlsafe_b64decode(payload))
    return int(claims["exp"] - time.time())

## Sign in (device code)
Run the cell, open the link, approve — it returns once you do.

In [ ]:
from pylakekeeper import DeviceCodeFlow

try:
    from IPython.display import HTML, display

    def notebook_prompt(p):
        target = p.verification_uri_complete or p.verification_uri
        html = (
            '<div style="padding:12px;border:1px solid #888;border-radius:8px;'
            'font-family:sans-serif;max-width:520px">'
            "<b>🔐 Sign in to continue</b><br>"
            f'Open <a href="{target}" target="_blank">{target}</a>'
        )
        if not p.verification_uri_complete and p.user_code:
            html += f' and enter code <code style="font-size:1.2em">{p.user_code}</code>'
        html += "</div>"
        display(HTML(html))
except ImportError:
    notebook_prompt = None

auth = DeviceCodeFlow(
    device_authorization_url=DEVICE_AUTHORIZATION_URL,
    token_url=TOKEN_URL,
    client_id=CLIENT_ID,
    scope=SCOPE,
    prompt=notebook_prompt,
)
print("Signed in ✅  token expires in", token_seconds_left(auth.auth_header()), "s")

## Sample data

In [ ]:
import numpy as np
import pyarrow as pa

EMBED_DIM = 8
ROWS = 100
_rng = np.random.default_rng(42)
_embeddings = _rng.standard_normal((ROWS, EMBED_DIM)).astype(np.float32)

sample = pa.table(
    {
        "id": pa.array(range(ROWS), type=pa.int64()),
        "sku": pa.array([f"SKU-{i:06d}" for i in range(ROWS)]),
        "embedding": pa.FixedSizeListArray.from_arrays(
            pa.array(_embeddings.reshape(-1), type=pa.float32()), EMBED_DIM
        ),
    }
)
print(sample.schema)

## Create the Lance table and write it

`load(..., vended=True)` returns the base location **and** short-lived S3 credentials.
`resp.lance_storage_options` maps them straight into Lance's object-store options.

In [ ]:
import lance

from pylakekeeper import Client, ConflictError, GenericTableFormat

TABLE = os.environ.get("TABLE", "image_embeddings")

with Client(base_url=LAKEKEEPER, warehouse=WAREHOUSE_ID, auth=auth, project_id=PROJECT_ID) as c:
    try:
        c.generic_tables.create(
            NAMESPACE,
            TABLE,
            format=GenericTableFormat.LANCE,
            properties={"embedding-dim": str(EMBED_DIM)},
        )
    except ConflictError:
        print(f"{NAMESPACE}.{TABLE} already exists — continuing")

    resp = c.generic_tables.load(NAMESPACE, TABLE, vended=True)
    print("location =", resp.location)
    lance.write_dataset(
        sample, resp.location, storage_options=resp.lance_storage_options, mode="overwrite"
    )

## Read the Lance table back

Reading is a separate step: re-load for **fresh** vended credentials (the earlier ones are
short-lived), then open with `lance.dataset`. Lance is columnar, so project only what you need.

In [ ]:
import lance

from pylakekeeper import Client

with Client(base_url=LAKEKEEPER, warehouse=WAREHOUSE_ID, auth=auth, project_id=PROJECT_ID) as c:
    resp = c.generic_tables.load(NAMESPACE, TABLE, vended=True)  # fresh short-lived creds

ds = lance.dataset(resp.location, storage_options=resp.lance_storage_options)
print("row_count =", ds.count_rows())
print(ds.schema)
print(ds.to_table(columns=["id", "sku"], limit=5))

## The session outlives the access token
Force an expiry — the refresh token renews silently, no second device prompt.

In [ ]:
before = token_seconds_left(auth.auth_header())
print("current access token expires in", before, "s")

auth.invalidate()  # what the transport does on a 401 — simulate expiry

after = token_seconds_left(auth.auth_header())  # no re-login: silent refresh
print("after refresh, new token expires in", after, "s")
assert after >= before, "expected a freshly-minted token"
print("\n✅ Session renewed via refresh_token — no re-authentication.")